In [1]:
# FORECAST - Week 3: Feature Engineering, Model, Backtest & Risk Scoring
# Builds on weekly_sales.csv from week 2. Trains a LightGBM model, backtests it against the seasonal-naive baseline, forecasts forward, and scores stockout/overstock risk.


In [2]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb

PROCESSED_DIR = Path("../data/processed")
weekly = pd.read_csv(PROCESSED_DIR / "weekly_sales.csv", parse_dates=["week_start"])
inventory_position = pd.read_csv(PROCESSED_DIR / "inventory_position.csv")
sku_master = pd.read_csv(PROCESSED_DIR / "sku_master.csv")


In [4]:
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")
all_skus = weekly["sku_id"].unique()
full_index = pd.MultiIndex.from_product([all_skus, all_weeks], names=["sku_id", "week_start"])
panel = (
    weekly.set_index(["sku_id","week_start"])[["units_sold", "revenue", "promo_flag"]]
    .reindex(full_index)
    .reset_index()                 
)
panel["units_sold"] = panel["units_sold"].fillna(0)
panel["promo_flag"] = panel["promo_flag"].fillna(False).astype(int)
panel = panel.sort_values(["sku_id", "week_start"]).reset_index(drop = True)

cat_map = weekly[["sku_id", "category", "subcategory"]].drop_duplicates("sku_id")
panel = panel.merge(cat_map, on="sku_id", how="left")
panel["category"] = panel["category"]. astype("category")
panel.head()


,sku_id,week_start,units_sold,revenue,promo_flag,category,subcategory
0,SKU00001,2024-01-01,83,3902.71,1,Home & Kitchen,Cookware
1,SKU00001,2024-01-08,24,1027.18,1,Home & Kitchen,Cookware
2,SKU00001,2024-01-15,30,1382.77,1,Home & Kitchen,Cookware
3,SKU00001,2024-01-22,58,2758.85,1,Home & Kitchen,Cookware
4,SKU00001,2024-01-29,10,481.76,0,Home & Kitchen,Cookware


In [5]:
# All features use shift(1) or later, so no feature ever sees the current
# week's own target value — this is what "no leakage" means in practice.
g = panel.groupby("sku_id")["units_sold"]

for lag in [1, 2, 4, 8, 52]:
    panel[f"lag_{lag}"] = g.shift(lag)

for window in [4, 8]:
    panel[f"roll_mean_{window}"] = g.shift(1).rolling(window).mean().reset_index(level=0, drop=True)
    panel[f"roll_std_{window}"] = g.shift(1).rolling(window).std().reset_index(level=0, drop=True)

panel["week_of_year"] = panel["week_start"].dt.isocalendar().week.astype(int)
panel["month"] = panel["week_start"].dt.month

FEATURES = [c for c in panel.columns if c.startswith("lag_") or c.startswith("roll_")] + \
           ["week_of_year", "month", "promo_flag", "category"]
TARGET = "units_sold"

print(f"{len(FEATURES)} features:", FEATURES)
print("Rows with all features present:", panel.dropna(subset=FEATURES).shape[0], "/", len(panel))

13 features: ['lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_52', 'roll_mean_4', 'roll_std_4', 'roll_mean_8', 'roll_std_8', 'week_of_year', 'month', 'promo_flag', 'category']
Rows with all features present: 13250 / 26250


In [6]:
def wape(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    return np.abs(actual - forecast).sum() / actual.sum()

In [7]:
# Same rolling-origin idea as Week 2's baseline backtest - train only on data
# before each origin, score only on the weeks after it. This is what makes the comparsion to the baseline fair.
FORECAST_HORIZON = 6
max_week = panel["week_start"].max()
latest_possible_origin = max_week - pd.Timedelta(weeks=FORECAST_HORIZON)
origin_weeks = pd.date_range(end=latest_possible_origin, periods=6, freq="4W-MON")
results = []
for origin in origin_weeks:
    train = panel[panel["week_start"] <= origin].dropna(subset=FEATURES + [TARGET])
    horizon_end = origin + pd.Timedelta(weeks=FORECAST_HORIZON)
    test = panel[(panel["week_start"] > origin) & (panel["week_start"] <= horizon_end)].dropna(subset=FEATURES)

    if train.empty or test.empty:
        continue
    model = lgb.LGBMRegressor(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        min_child_samples=20, verbose=-1, random_state=42
    )
    model.fit(train[FEATURES], train[TARGET], categorical_feature=["category"])
    test = test.copy()
    test["pred"] = model.predict(test[FEATURES]).clip(min=0)
    test = test.dropna(subset=[TARGET])
    score = wape(test[TARGET], test["pred"])
    results.append({"origin_week":origin.date(), "n_scored": len(test), "waped_model": round(score, 4)})

backtest_results = pd.DataFrame(results)
backtest_results

,origin_week,n_scored,waped_model
0,2025-06-30,1500,0.2465
1,2025-07-28,1500,0.2300
2,2025-08-25,1500,0.2273
3,2025-09-22,1500,0.2713
4,2025-10-20,1500,0.3821
5,2025-11-17,1500,0.2304


In [8]:
baseline_backtest = pd.read_csv(PROCESSED_DIR / "baseline_forecast.csv", parse_dates=["week_start"])
# recompute baseline WAPE the same way Week 2 did, so the comparison is apples-to-apples
baseline_wape = wape(
    baseline_backtest.dropna(subset=["forecast_naive"])["units_sold"],
    baseline_backtest.dropna(subset=["forecast_naive"])["forecast_naive"],
)

model_wape = backtest_results["waped_model"].mean()
print(f"Baseline WAPE (seasonal-naive): {baseline_wape:.4f}")
print(f"Model WAPE (LightGBM):          {model_wape:.4f}")
print(f"Improvement: {(1 - model_wape/baseline_wape)*100:.1f}%")

if model_wape < baseline_wape:
    print("\n Model beats the baseline — safe to ship.")
else:
    print("\n Model did NOT beat the baseline — ship the baseline instead and report this honestly (brief Section 07).")

Baseline WAPE (seasonal-naive): 0.3215
Model WAPE (LightGBM):          0.2646
Improvement: 17.7%

 Model beats the baseline — safe to ship.


In [9]:
# Once the backtest has proven the model beats the baseline, retrain on
# ALl history (not held back like the backtest folds) to get the best
# possible model for actual forecasting forward.

train_full = panel.dropna(subset=FEATURES + [TARGET])
final_model = lgb.LGBMRegressor(
    n_estimators=200, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, verbose=-1, random_state=42
)
final_model.fit(train_full[FEATURES], train_full[TARGET], categorical_feature=["category"])
print(f"Trained on {len(train_full):,} rows")

Trained on 13,250 rows


In [10]:
# Forecasting forward is different from backtesting: for week+2, lag_1 needs
# week+1's value — which doesn't exist yet. So we predict one week at a time
# and feed each prediction back in as if it were real history ("recursive
# forecasting"). This is a standard technique, not a shortcut — just be
# aware errors can compound over the horizon, which is why intervals matter
# (see brief Section 07.2 on the stretch goal).
last_week = panel["week_start"].max()
future_weeks = pd.date_range(last_week + pd.Timedelta(weeks=1), periods=FORECAST_HORIZON, freq="W-MON")

forecasts = []
for sku_id, hist in panel.groupby("sku_id"):
    hist = hist.sort_values("week_start")
    series = hist["units_sold"].tolist()
    category = hist["category"].iloc[0]

    for wk in future_weeks:
        s = pd.Series(series)
        feat = {
            "lag_1": s.iloc[-1] if len(s) >= 1 else np.nan,
            "lag_2": s.iloc[-2] if len(s) >= 2 else np.nan,
            "lag_4": s.iloc[-4] if len(s) >= 4 else np.nan,
            "lag_8": s.iloc[-8] if len(s) >= 8 else np.nan,
            "lag_52": s.iloc[-52] if len(s) >= 52 else np.nan,
            "roll_mean_4": s.iloc[-4:].mean() if len(s) >= 4 else np.nan,
            "roll_std_4": s.iloc[-4:].std() if len(s) >= 4 else np.nan,
            "roll_mean_8": s.iloc[-8:].mean() if len(s) >= 8 else np.nan,
            "roll_std_8": s.iloc[-8:].std() if len(s) >= 8 else np.nan,
            "week_of_year": wk.isocalendar().week,
            "month": wk.month,
            "promo_flag": 0,  # no promo calendar available for future weeks — documented assumption
            "category": category,
        }
        X = pd.DataFrame([feat])
        X["category"] = X["category"].astype(panel["category"].dtype)
        pred = max(0, final_model.predict(X[FEATURES])[0])
        forecasts.append({"sku_id": sku_id, "week_start": wk, "forecast": pred})
        series.append(pred)  # feed the prediction back in for the next step

forecast_df = pd.DataFrame(forecasts)
forecast_df.to_csv(PROCESSED_DIR / "forward_forecast.csv", index=False)
forecast_df.head(10)

,sku_id,week_start,forecast
0,SKU00001,2026-01-05,25.459913
1,SKU00001,2026-01-12,32.556224
2,SKU00001,2026-01-19,32.533196
3,SKU00001,2026-01-26,32.195784
4,SKU00001,2026-02-02,37.098553
5,SKU00001,2026-02-09,34.522791
6,SKU00002,2026-01-05,29.599285
7,SKU00002,2026-01-12,23.396652
8,SKU00002,2026-01-19,27.663678
9,SKU00002,2026-01-26,31.511150


In [11]:
# inventory.csv has no lead_time_days column (see Week 1 data-quality notes),
# so lead time is an explicit, documented assumption rather than a real value.
ASSUMED_LEAD_TIME_WEEKS = 4
OVERSTOCK_THRESHOLD_WEEKS = 12

horizon_forecast = (
    forecast_df.groupby("sku_id", as_index=False)["forecast"]
    .sum()
    .rename(columns={"forecast": "forecast_horizon_units"})
)

risk = horizon_forecast.merge(inventory_position, on="sku_id", how="left")
risk = risk.merge(sku_master[["sku_id", "unit_price", "list_price"]], on="sku_id", how="left")

risk["weekly_avg_forecast"] = risk["forecast_horizon_units"] / FORECAST_HORIZON
risk["weeks_of_cover"] = risk["stock_on_hand"] / risk["weekly_avg_forecast"].replace(0, np.nan)

# stockout risk: how much of the "safe" lead-time window is NOT covered by stock
risk["stockout_risk"] = (1 - risk["weeks_of_cover"] / ASSUMED_LEAD_TIME_WEEKS).clip(0, 1)
# overstock risk: how far weeks_of_cover exceeds a reasonable holding threshold
risk["overstock_risk"] = ((risk["weeks_of_cover"] - OVERSTOCK_THRESHOLD_WEEKS) / OVERSTOCK_THRESHOLD_WEEKS).clip(0, 1)
risk[["stockout_risk", "overstock_risk"]] = risk[["stockout_risk", "overstock_risk"]].fillna(0)

risk.head()

,sku_id,forecast_horizon_units,stock_on_hand,reorder_point,safety_stock,n_stores_carrying,last_restock_date,category,subcategory,unit_price,list_price,weekly_avg_forecast,weeks_of_cover,stockout_risk,overstock_risk
0,SKU00001,194.366462,215.0,100.0,33.0,1.0,2025-12-22,Home & Kitchen,Cookware,619.77,813.41,32.394410,6.636947,0.0,0.000000
1,SKU00002,187.058455,0.0,1582.0,573.0,25.0,2025-10-06,Dairy & Bakery,Bread,49.57,70.38,31.176409,0.000000,1.0,0.000000
2,SKU00003,185.280634,414.0,250.0,85.0,4.0,2025-12-13,Stationery & Office,Notebooks,83.67,151.28,30.880106,13.406690,0.0,0.117224
3,SKU00004,184.442130,148.0,99.0,35.0,1.0,2025-12-03,Dairy & Bakery,Eggs,156.32,233.64,30.740355,4.814518,0.0,0.000000
4,SKU00005,200.695916,411.0,284.0,104.0,5.0,2025-12-26,Home Care,Pest Control,83.46,138.50,33.449319,12.287246,0.0,0.023937


In [12]:
def quadrant(row):
    if row["stockout_risk"] >= 0.5 and row["overstock_risk"] >= 0.5:
        return "Watch / Volatile"
    if row["stockout_risk"] >= 0.5:
        return "Reorder Now"
    if row["overstock_risk"] >= 0.5:
        return "Markdown / Clear"
    return "Healthy"

risk["quadrant"] = risk.apply(quadrant, axis=1)
print(risk["quadrant"].value_counts())


quadrant
Healthy             191
Reorder Now          34
Markdown / Clear     25
Name: count, dtype: int64


In [13]:
shortfall_units = (risk["weekly_avg_forecast"] * ASSUMED_LEAD_TIME_WEEKS - risk["stock_on_hand"]).clip(lower=0)
excess_units = (risk["stock_on_hand"] - risk["weekly_avg_forecast"] * OVERSTOCK_THRESHOLD_WEEKS).clip(lower=0)

risk["rupee_at_risk_stockout"] = (shortfall_units * risk["list_price"]).round(2)
risk["rupee_locked_overstock"] = (excess_units * risk["unit_price"]).round(2)

print("Total revenue at risk from stockouts: ₹", risk["rupee_at_risk_stockout"].sum())
print("Total capital locked in overstock:    ₹", risk["rupee_locked_overstock"].sum())

risk.to_csv(PROCESSED_DIR / "risk_scoring.csv", index=False)
print("\nSaved risk_scoring.csv")

Total revenue at risk from stockouts: ₹ 2943775.97
Total capital locked in overstock:    ₹ 77092325.91

Saved risk_scoring.csv
